# Zero to Verifiable
### Teach a small model to solve Countdown: SFT shows one solution, GRPO rewards any

**The puzzle:** combine all the given numbers, each exactly once, with `+ − × ÷` to hit a target. `14 18 3 → 68` is solved by `3*18=54`, then `14+54=68`. Finding a solution is a search; checking one takes a few lines of code. That gap is what reinforcement learning with a verifier exploits.

SFT shows the model **one** solution per puzzle. Most puzzles have several, and GRPO rewards **any** correct one the model finds by sampling.

You type the parts that are the lesson: the checker, the label mask, the reward and the three configs. The `z2v` helper package does the rest (loading, batching, timing, plotting); each helper is a short function you can read.

| | |
|---|---|
| **Model** | `Qwen/Qwen2.5-0.5B-Instruct` (revision-pinned), LoRA r=16 on 16-bit weights |
| **GPU** | `Runtime ▸ Change runtime type ▸ T4 GPU`: ~4 min SFT + ~12 min GRPO + ~3 min evaluation |
| **Live** | TensorBoard (step 1): `train/loss`, `eval/exact_match_sft`, `train/reward`, `train/frac_reward_zero_std`, `eval/exact_match_rl` |

**If a function you typed breaks**, run `from z2v.reference import check, encode, reward` and carry on with the reference versions. **If you fall behind**, open the complete notebook, `notebooks/workshop.ipynb` in the same repository.

Cells marked ⏱ take minutes: start them, then switch to the slides.

## 1 · Setup

Installs the helper, which pins `transformers`, `trl` and `peft` (Colab's own `torch` stays). `setup()` checks the GPU, seeds everything, downloads the model weights (~1 GB) and opens TensorBoard, which stays empty until training logs its first step.

In [ ]:
!pip install -q "git+https://github.com/pratyush-acharya/zero-to-verifiable@72e2c12c0aa95481cf6e6688b70b19b0f50d7e91"
# Type: from z2v import *   then   setup()   (2 lines)

## 2 · Puzzles and the checker

**Rules.** Use every number exactly once; each step combines two available numbers with `+ − × ÷`; every intermediate result must be a positive whole number. An answer is one step per line, `a op b = c`, ending at the target.

**Puzzles.** Pick 3 or 4 numbers from 1–20, combine them in a random legal order, and keep the result as the target if it lands between 10 and 100. That random path is the one worked solution SFT gets to see. One seed, so everyone gets the same puzzles, and no puzzle appears in two splits:

| split | 3 numbers | 4 numbers | used for |
|---|---|---|---|
| `train` | 5,000 | 11,000 | SFT worked examples |
| `rl` | 1,000 | 3,000 | GRPO prompts, scored only by the checker |
| `dev` | 32 | 32 | the live accuracy curves |
| `test` | 150 | 150 | the final table, never trained on |
| `bigger` | 30 | 30 | numbers up to 50 instead of 20 |

In [ ]:
# Type: puzzles = make_puzzles(), then show_puzzles(puzzles)   (2 lines)

**The checker** replays an answer step by step. `STEP.findall(text)` finds every `a op b = c`; `apply(a, op, b)` is the rules: it returns the result, or `None` if the step is illegal. `check` returns `(legal?, final value)`.

In [ ]:
# Type: def check(text, nums) -> (legal?, final value).
#       Replay each step from STEP.findall(text): both numbers must still be in the pool,
#       apply(a, op, b) must equal c, and the result goes back in the pool.
#       Legal only if exactly one number is left at the end.   (~12 lines)

**The floor.** A small solver lists every legal way to combine a puzzle's numbers. Picking one of those paths at random hits the target only a few percent of the time: that is the number a model has to beat.

In [ ]:
# Type: floor(puzzles)   (1 line)

## 3 · Text to tokens: the mask

One system turn, one user turn, one assistant answer. `render(p)` is the chat text with the assistant turn left open (what the model sees when it answers); `render(p, answer)` adds the worked solution; `ids(text)` turns text into token ids. The label mask is the teaching point: **every token before the answer gets the label `-100`**, so the loss is computed only on the steps the model must produce. Training and generation use the same `render()`.

In [ ]:
# Type: def encode(p): token ids of render(p) and of render(p, p["answer"]);
#       labels = -100 for every prompt token, then the answer's ids;
#       return input_ids, labels and attention_mask. Then show_mask(encode(puzzles.train[1]))   (~6 lines)

## 4 · The model and LoRA

Qwen2.5-0.5B in 16-bit is about 1 GB. LoRA freezes those weights and trains a small low-rank update beside each linear layer; `load_model` prints how many parameters that is.

In [ ]:
# Type: lora = LoraConfig(r=16, lora_alpha=16, target_modules="all-linear", lora_dropout=0.0)
#       model = load_model("Qwen/Qwen2.5-0.5B-Instruct", lora)   (2 lines, or 7 with one knob per line)

## 5 · Before training

Measure the starting point; don't assume it. Two measurements, every time:
- **greedy**: the model's single most likely answer. This is what "accuracy" usually means.
- **8 samples at temperature 1**: how GRPO sees the model. The share of puzzles where **any of 8** is right is the room RL has to work with, because RL can only reinforce answers the model sometimes produces.

`legal steps` counts answers whose arithmetic is all true and that use every number, whether or not they hit the target. `adapter=False` switches the LoRA adapter off: these are the base weights.

`explain_sft` shows what SFT will push on: the probability the model gives each token of one worked solution, and the loss for that example, the sum of −log p.

In [ ]:
# Type: base = evaluate(model, puzzles, check, adapter=False)
#       explain_sft(model, puzzles.train[1], encode)   (2 lines)

## 6 · SFT

One worked solution per puzzle, 32 per step. `train_sft` first times one real step, then sets `max_steps` so the run fits the time budget: a slower GPU trains less instead of overrunning the session, and a wall-clock stop is the backstop. To your config it adds only plumbing: that `max_steps`, TensorBoard logging every 5 steps, 16-bit, no checkpoints, seed 3407.

⏱ **~4 min: start it, then switch to the slides.** In TensorBoard, `train/loss` should fall fast, then flatten, and `eval/exact_match_sft` (greedy, 64 dev puzzles, every 25 steps) should rise well above the floor. The last line repeats `explain_sft` on the same example.

In [ ]:
# Type: sft_cfg = TrainingArguments(learning_rate=2e-4, per_device_train_batch_size=32,
#                                   lr_scheduler_type="linear", warmup_steps=20)
#       train_sft(model, puzzles.train, encode, sft_cfg, minutes=4)      <- ⏱ ~4 min
#       explain_sft(model, puzzles.train[1], encode)   (~8 lines)

## 7 · Room for RL

Same puzzles, same renderer, now with the SFT adapter. Compare the two columns: if **any of 8** is well above **greedy**, the model already finds solutions it doesn't commit to, and that gap is what GRPO can turn into greedy accuracy. If the two are close, RL has little to work with, and that is worth saying out loud.

In [ ]:
# Type: sft = evaluate(model, puzzles, check), then room_for_rl(base, sft)   (2 lines)

## 8 · The reward

GRPO needs a number for every sampled answer, and the checker provides it:
- **1.0** for a legal answer that hits the target, *whichever* solution it is;
- **0.5 × 0.9^|off|** for a legal answer that misses (0.45 at one off, 0.17 at ten off);
- **0** for anything else: false arithmetic, a number used twice or left out, or no steps.

GRPO calls the reward with the dataset's columns by name (`nums`, `target`, plus extras that `**kw` absorbs), and each completion is a list of chat messages, so the text is `c[-1]["content"]`. `reward_ladder` calls your reward exactly that way. Ask the room: how would you cheat this reward? (Writing `13+20=98` to "hit" the target earns nothing.)

In [ ]:
# Type: def reward(completions, nums, target, **kw) -> one score per completion:
#       the text is c[-1]["content"]; 0.0 if not legal, 1.0 if it hits the target,
#       else 0.5 * 0.9 ** abs(final - target). Then reward_ladder(reward, puzzles.rl)   (~10 lines)

## 9 · GRPO

GRPO samples a **group** of 8 answers to each puzzle from the 4,000 RL puzzles SFT never saw, scores each with your reward, and compares each answer with the rest of its group: answers above the group's mean are made more likely, those below less likely. If all 8 score the same, the puzzle teaches nothing; `train/frac_reward_zero_std` in TensorBoard is the share of such puzzles.

`explain_grpo` works through one group by hand on a live puzzle. `train_grpo` adds plumbing (TensorBoard, 16-bit, no checkpoints) and, for knobs left at the library default, the validated values: rewards scaled by the group's std, gradient clipping at 0.1, a constant learning rate after 10 warm-up steps, logging every step.

In [ ]:
# Type: grpo_cfg = GRPOConfig(num_generations=8, temperature=1.0, learning_rate=2e-5, beta=0.0,
#                             max_completion_length=64, per_device_train_batch_size=32,
#                             gradient_accumulation_steps=2)
#       explain_grpo(model, puzzles.rl[0], reward, grpo_cfg)   (~10 lines)

⏱ **~12 min: start it, then switch to the slides.** In TensorBoard, `train/reward` should climb and `train/frac_reward_zero_std` should stay well below 1. `train/completions/mean_length` should stay near the length of a worked solution: shrinking can mean the model has found a shortcut the checker allows, which is worth a look at the samples.

In [ ]:
# Type: train_grpo(model, puzzles.rl, reward, grpo_cfg, minutes=12)      <- ⏱ ~12 min   (1 line)

## 10 · Results

One model and one renderer; only the adapter changes: **base** (adapter off) → **SFT** → **SFT→RL** (GRPO continued the SFT adapter). The first row is the floor: a random legal path through each puzzle.

Puzzles are independent, so the 95% intervals resample the 300 test puzzles, and the differences are paired (the same puzzles in both arms). `bigger` uses numbers up to 50 (never trained on); `seen` and `rl puzzles` are 40-puzzle spot checks from the SFT and RL sets.

Read the last two columns together. **Greedy** and **avg of 8** measure how reliably the model solves a puzzle; **any of 8** measures how many puzzles it can solve at all with several tries. If RL raises the first two and lowers the third, it has turned attempts that sometimes work into answers the model commits to, and explores less. The last printed line holds this run's numbers for the slides.

In [ ]:
# Type: rl = evaluate(model, puzzles, check), then results(base, sft, rl)   (2 lines)

**What did RL change in the answers?** The checker only cares whether an answer is legal and hits the target, so RL is free to change *how* the model solves puzzles. The table compares the operators in the worked solutions SFT imitated with those in each model's legal answers on the test puzzles. If RL leans on `+` and `−`, easy moves have become safer bets: they are legal more often, and near-misses still earn partial reward. Is that a skill or a shortcut?

In [ ]:
# Type: operator_mix(sft, rl)   (1 line)

## Read TensorBoard, then say only what the numbers allow

| scalar | what it should do | what it means if it doesn't |
|---|---|---|
| `train/loss` (sft run) | falls fast, then flattens | flat from step 0 ⇒ the mask or the learning rate is wrong |
| `eval/exact_match_sft` | rises well above the random-path floor | stuck near 0 ⇒ training and generation render differently |
| `train/reward` (grpo run) | climbs | flat ⇒ check `frac_reward_zero_std` |
| `train/frac_reward_zero_std` | well below 1 | near 1 ⇒ the 8 answers to a puzzle all score the same: nothing to compare |
| `train/completions/mean_length` | steady, near a worked solution | shrinking or growing fast ⇒ look at the samples |
| `eval/exact_match_rl` | rises | falling ⇒ RL is hurting; lower the learning rate |

**Claims these numbers do not support:**
- "RL taught it to search": compare **any of 8** before RL with **greedy** after. RL mostly turns answers the model could already sample into answers it commits to.
- "RL made it better at everything": check **any of 8**. Committing more usually means exploring less.
- "It learned arithmetic": the **legal** column says how often its steps are true; the **bigger** column says whether that holds for numbers it never saw.
- "RL beat SFT", when the interval crosses zero. And even when it doesn't, we didn't test spending the same 12 minutes on more SFT.